# SurfaceTarget-CRC — phase 01 — patch 1 11

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.11
#
# RNA-seq Duplicate File and Biospecimen Investigation
#
# Objectives:
# 1. Identify patients with multiple tumor samples/files
# 2. Retrieve detailed file metadata from GDC
# 3. Investigate sample and aliquot identifiers
# 4. Examine workflow versions and file timestamps
# 5. Compare sample identifiers with MSI annotations
# 6. Save findings for reproducible sample selection
#
# IMPORTANT:
# No RNA-seq files are selected or downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

CANDIDATE_PATH = (
    METADATA_DIR /
    "tcga_candidate_tumor_rnaseq_files.csv"
)

COHORT_PATH = (
    METADATA_DIR /
    "tcga_mss_evidence_tiers.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.11 — RNA-SEQ DUPLICATE INVESTIGATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load candidate files and cohort information
# ------------------------------------------------------------

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype=str,
    keep_default_na=False
)

cohort = pd.read_csv(
    COHORT_PATH,
    dtype=str
)

print("\nCandidate RNA-seq files:", len(candidates))

print(
    "Candidate patients:",
    candidates["patient_id"].nunique()
)

# ------------------------------------------------------------
# 3. Identify patients requiring review
# ------------------------------------------------------------

patient_summary = (
    candidates.groupby("patient_id")
    .agg(
        n_samples=("sample_id", "nunique"),
        n_files=("file_id", "nunique")
    )
    .reset_index()
)

review_patients = patient_summary[
    (patient_summary["n_samples"] > 1)
    | (patient_summary["n_files"] > 1)
].copy()

print("\n" + "-" * 65)
print("PATIENTS REQUIRING REVIEW")
print("-" * 65)

print(
    review_patients.to_string(index=False)
)

review_ids = review_patients["patient_id"].tolist()

if not review_ids:
    print("\nNo duplicate cases detected.")
else:
    print(
        "\nPatients requiring investigation:",
        len(review_ids)
    )

# ------------------------------------------------------------
# 4. Extract all candidate files for these patients
# ------------------------------------------------------------

review_files = candidates[
    candidates["patient_id"].isin(review_ids)
].copy()

file_ids = sorted(
    review_files["file_id"].unique().tolist()
)

print("\nFiles requiring metadata investigation:", len(file_ids))

print("\nOriginal inventory:")

print(
    review_files[
        [
            "patient_id",
            "sample_id",
            "file_id",
            "file_name"
        ]
    ].sort_values(
        ["patient_id", "sample_id", "file_id"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Retrieve detailed GDC file metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING GDC FILE PROVENANCE")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "created_datetime",
    "updated_datetime",
    "analysis.workflow_type",
    "analysis.workflow_version",
    "analysis.input_files.file_id",
    "cases.submitter_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type",
    "cases.samples.portions.analytes.aliquots.submitter_id"
]

filters = {
    "op": "in",
    "content": {
        "field": "file_id",
        "value": file_ids
    }
}

params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": 100
}

if not file_ids:
    raise RuntimeError(
        "No ambiguous files found. "
        "No duplicate investigation is required."
    )

response = session.get(
    f"{GDC_API}/files",
    params=params,
    timeout=90
)

response.raise_for_status()

payload = response.json()

hits = payload["data"]["hits"]

print("\nFiles requested:", len(file_ids))
print("Files retrieved:", len(hits))

returned_ids = {
    item["file_id"]
    for item in hits
}

missing_ids = set(file_ids) - returned_ids

if missing_ids:
    raise RuntimeError(
        f"GDC did not return these file IDs: {missing_ids}"
    )

print("\nFile retrieval validation: PASSED")

# ------------------------------------------------------------
# 6. Save original GDC metadata response
# ------------------------------------------------------------

raw_metadata_path = (
    METADATA_DIR /
    "tcga_duplicate_files_gdc_raw_metadata.json"
)

raw_metadata = {
    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": "NCI Genomic Data Commons",
    "endpoint": f"{GDC_API}/files",
    "requested_file_ids": file_ids,
    "fields": fields,
    "records": hits
}

with open(raw_metadata_path, "w") as f:
    json.dump(raw_metadata, f, indent=4)

# ------------------------------------------------------------
# 7. Extract biospecimen and aliquot information
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING BIOSPECIMEN INFORMATION")
print("-" * 65)

records = []

for item in hits:

    file_id = item.get("file_id")

    analysis = item.get("analysis") or {}

    input_files = analysis.get("input_files") or []

    input_file_ids = sorted({
        f.get("file_id")
        for f in input_files
        if f.get("file_id")
    })

    cases = item.get("cases") or []

    for case in cases:

        patient_id = case.get("submitter_id")

        samples = case.get("samples") or []

        sample_ids = set()
        sample_types = set()
        aliquot_ids = set()

        for sample in samples:

            sample_id = sample.get("submitter_id")

            if sample_id:
                sample_ids.add(sample_id)

            sample_type = sample.get("sample_type")

            if sample_type:
                sample_types.add(sample_type)

            portions = sample.get("portions") or []

            for portion in portions:

                analytes = portion.get("analytes") or []

                for analyte in analytes:

                    aliquots = analyte.get("aliquots") or []

                    for aliquot in aliquots:

                        aliquot_id = aliquot.get("submitter_id")

                        if aliquot_id:
                            aliquot_ids.add(aliquot_id)

        records.append({
            "patient_id": patient_id,
            "file_id": file_id,
            "file_name": item.get("file_name"),
            "file_size_bytes": item.get("file_size"),
            "md5sum": item.get("md5sum"),
            "created_datetime": item.get("created_datetime"),
            "updated_datetime": item.get("updated_datetime"),
            "workflow_type": analysis.get("workflow_type"),
            "workflow_version": analysis.get("workflow_version"),
            "input_file_ids": "; ".join(input_file_ids),
            "n_input_files": len(input_file_ids),
            "sample_ids": "; ".join(sorted(sample_ids)),
            "sample_types": "; ".join(sorted(sample_types)),
            "n_samples": len(sample_ids),
            "aliquot_ids": "; ".join(sorted(aliquot_ids)),
            "n_aliquots": len(aliquot_ids)
        })

details = pd.DataFrame(records)

if details.empty:
    raise RuntimeError(
        "No biospecimen metadata could be extracted."
    )

# ------------------------------------------------------------
# 8. Display detailed results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DETAILED FILE AND BIOSPECIMEN AUDIT")
print("-" * 65)

for patient_id in review_ids:

    print("\n" + "=" * 60)
    print("PATIENT:", patient_id)
    print("=" * 60)

    subset = details[
        details["patient_id"] == patient_id
    ].sort_values("file_id")

    for _, row in subset.iterrows():

        print("\nFile ID:", row["file_id"])

        print("File:", row["file_name"])

        print("Samples:", row["sample_ids"])

        print("Sample Types:", row["sample_types"])

        print("Aliquots:", row["aliquot_ids"])

        print("Workflow:", row["workflow_type"])

        print("Workflow Version:", row["workflow_version"])

        print("Input File IDs:", row["input_file_ids"])

        print("Created:", row["created_datetime"])

        print("Updated:", row["updated_datetime"])

        print("MD5:", row["md5sum"])

        print("-" * 50)

# ------------------------------------------------------------
# 9. Compare with existing RNA-seq inventory
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL INVENTORY COMPARISON")
print("-" * 65)

for file_id in file_ids:

    original_samples = set(
        review_files.loc[
            review_files["file_id"] == file_id,
            "sample_id"
        ]
    )

    current_rows = details[
        details["file_id"] == file_id
    ]

    current_samples = set()

    for value in current_rows["sample_ids"]:

        for sample_id in str(value).split(";"):

            sample_id = sample_id.strip()

            if sample_id:
                current_samples.add(sample_id)

    print("\nFile:", file_id)

    print(
        "Original inventory samples:",
        sorted(original_samples)
    )

    print(
        "Current GDC samples:",
        sorted(current_samples)
    )

    if original_samples == current_samples:

        print("Metadata correspondence: MATCH")

    else:

        print("Metadata correspondence: REVIEW REQUIRED")

# ------------------------------------------------------------
# 10. Compare with MSI sample identifiers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI SAMPLE IDENTIFIER COMPARISON")
print("-" * 65)

msi_columns = [
    "patientId",
    "gdc_project",
    "msi_group",
    "mantis_sample_id",
    "msisensor_sample_id",
    "published_sample_id",
    "evidence_tier"
]

msi_review = cohort[
    cohort["patientId"].isin(review_ids)
][msi_columns]

print("\n")
print(
    msi_review.to_string(index=False)
)

# ------------------------------------------------------------
# 11. Save detailed audit
# ------------------------------------------------------------

details_path = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

details.to_csv(
    details_path,
    index=False
)

msi_review_path = (
    METADATA_DIR /
    "tcga_duplicate_patients_msi_sample_review.csv"
)

msi_review.to_csv(
    msi_review_path,
    index=False
)

# ------------------------------------------------------------
# 12. Save methodology notes
# ------------------------------------------------------------

methodology = {
    "project": "SurfaceTarget-CRC",
    "patch": "1.11",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "patients_reviewed": review_ids,
    "files_reviewed": len(file_ids),
    "source": "NCI GDC API",
    "file_selection_completed": False,
    "notes": [
        "Multiple files were investigated using GDC metadata.",
        "File size alone is not a valid selection criterion.",
        "Timestamps alone do not establish biological superiority.",
        "Sample and aliquot relationships require interpretation.",
        "No duplicate RNA-seq files have been selected or removed."
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_duplicate_file_investigation_methodology.json"
)

with open(methodology_path, "w") as f:
    json.dump(methodology, f, indent=4)

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.11 SUMMARY")
print("=" * 65)

print("\nPatients investigated:", len(review_ids))

print("Files investigated:", len(file_ids))

print("\nFiles saved:")

print("1. tcga_duplicate_files_gdc_raw_metadata.json")
print("2. tcga_duplicate_rnaseq_provenance_audit.csv")
print("3. tcga_duplicate_patients_msi_sample_review.csv")
print("4. tcga_duplicate_file_investigation_methodology.json")

print("\nNo expression data downloaded.")
print("No duplicate files selected.")

print("\nPATCH 1.11 COMPLETE")
print("=" * 65)